In [2]:
#install/import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from tpot import TPOTClassifier
import requests
import json
from tqdm.notebook import tqdm  # nice progress bars
from concurrent.futures import ThreadPoolExecutor
import io
import os
from pathlib import Path
import gzip

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\stopit\__init__.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""
Project: creating a model that can predict or perform learning to detect if a particular NADPH rate or level
will likely produce a cancer risk or reaction in people
Research shows GSH and TXN increases antioxidants production, whereas NADPH does the opposite
What model? Still not decided yet. Maybe classifier? To see which level of NADPH of a tissue indicates severity of cancer?
"""


#retrieve databases
clinical = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/clinical.tsv", sep='\t')
exposure = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/exposure.tsv", sep='\t')
followup = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/follow_up.tsv", sep='\t')
aliquot = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/aliquot.tsv", sep='\t')
analyte = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/analyte.tsv", sep='\t')
sample = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/sample.tsv", sep='\t')
portion = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/portion.tsv", sep='\t')
#slide = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/slide.tsv", sep='\t')
details = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/pathology_detail.tsv", sep='\t')


C:\Users\user\AppData\Local\Temp\ipykernel_14580\699337770.py:10: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  clinical = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/clinical.tsv", sep='\t')


In [3]:
#cleaning data (removing na, removing columns not of interest)
clinic_removena = clinical.replace("'--", None).dropna(axis=1, how='all')
expose_removena = exposure.replace("'--", None).dropna(axis=1, how='all')
aliquot_removena = aliquot.replace("'--", None).dropna(axis=1, how='all')
details_removena = details.replace("'--", None).dropna(axis=1, how='all')
analyte_removena = analyte.replace("'--", None).dropna(axis=1, how='all')
sample_removena = sample.replace("'--", None).dropna(axis=1, how='all')
portion_removena = portion.replace("'--", None).dropna(axis=1, how='all')
print(aliquot.columns)


#selecting columns to merge and do further exploration, plus model creation
'''For aliquots, selecting selcted_normal_wxs and matched_normal_wxs to see if there are further analyses that could be made for somatic mutations when compared to their normal counterparts'''
clinic_merge = clinic_removena.loc[:,['cases.case_id', 'cases.disease_type', 
    'demographic.age_at_index', 'demographic.ethnicity','demographic.gender', 
    'diagnoses.ajcc_pathologic_stage', 'diagnoses.classification_of_tumor', 
    'diagnoses.diagnosis_is_primary_disease', 'diagnoses.laterality','diagnoses.morphology', 
    'diagnoses.primary_diagnosis','diagnoses.prior_malignancy','diagnoses.tumor_of_origin']]
aliquot_merge = aliquot_removena.loc[:,['samples.sample_id', 'portions.portion_id', 
    'analytes.analyte_id', 'aliquots.aliquot_id', 'aliquots.analyte_type',
    'aliquots.concentration','aliquots.selected_normal_wxs', 'aliquots.no_matched_normal_wxs']]
expose_merge = expose_removena.loc[:, ['cases.case_id', 'exposures.alcohol_history']]
portion_merge = portion_removena.loc[:,['cases.case_id', 'samples.sample_id', 'portions.is_ffpe',
    'portions.portion_id', 'portions.portion_number']]
analyte_merge = analyte_removena.loc[:,['analytes.analyte_id', 'analytes.analyte_type', 
    'analytes.analyte_type_id', 'analytes.a260_a280_ratio','analytes.experimental_protocol_type',
    'analytes.normal_tumor_genotype_snp_match',
    'analytes.ribosomal_rna_28s_16s_ratio','analytes.rna_integrity_number', 'samples.sample_id']]
sample_merge = sample_removena.loc[:,['samples.sample_id','samples.submitter_id','samples.sample_type', 
    'samples.sample_type_id', 'samples.composition', 'samples.specimen_type', 'samples.tissue_type', 
    'samples.specimen_type','samples.tumor_descriptor']]
path_det_merge = details_removena.loc[:,['cases.case_id', 'diagnoses.diagnosis_id',
    'pathology_details.lymph_nodes_positive','pathology_details.lymph_nodes_tested',
    'pathology_details.pathology_detail_id',]]

#performing a major merge and thereafter doing removal of duplicated columns
complete_merge = clinic_merge.merge(path_det_merge, on='cases.case_id', how='left').merge(portion_merge, on='cases.case_id', how='left').merge(sample_merge, on='samples.sample_id', how='left').merge(analyte_merge, on='samples.sample_id', how='left').merge(aliquot_merge, on='analytes.analyte_id', how='left').drop_duplicates(ignore_index=True)
print(complete_merge.shape)

Index(['project.project_id', 'cases.case_id', 'cases.submitter_id',
       'samples.sample_id', 'samples.submitter_id', 'portions.portion_id',
       'portions.submitter_id', 'analytes.analyte_id', 'analytes.submitter_id',
       'aliquots.aliquot_id', 'aliquots.aliquot_quantity',
       'aliquots.aliquot_volume', 'aliquots.amount', 'aliquots.analyte_type',
       'aliquots.analyte_type_id', 'aliquots.concentration',
       'aliquots.no_matched_normal_low_pass_wgs',
       'aliquots.no_matched_normal_targeted_sequencing',
       'aliquots.no_matched_normal_wgs', 'aliquots.no_matched_normal_wxs',
       'aliquots.selected_normal_low_pass_wgs',
       'aliquots.selected_normal_targeted_sequencing',
       'aliquots.selected_normal_wgs', 'aliquots.selected_normal_wxs',
       'aliquots.source_center', 'aliquots.state', 'aliquots.submitter_id'],
      dtype='object')
(43224, 44)


In [ ]:
#store 'global' variables?
#this are global variables for retrieving relevant file id details
perused_caseids = []
file_data_SM = pd.DataFrame()
file_data_CNV = pd.DataFrame()
file_data_FSE = pd.DataFrame()
indices = 0

In [ ]:
#getting the particular gene expressions and sequences out from database
import requests
import pandas as pd
import io

#retrieving the sample submitter ids to be fit into the cancer bio portal
sample_barcode = complete_merge['samples.submitter_id'].drop_duplicates(ignore_index=True)

def get_cbioportal_sample_data(barcode, study_id="brca_tcga_pub"):
    """
    Fetch molecular data for a specific TCGA sample from cBioPortal.

    Parameters:
        barcode (str): TCGA barcode, e.g., 'TCGA-A7-A3IZ-01A-11R'
        study_id (str): Study ID in cBioPortal, default is TCGA Breast Cancer

    Returns:
        dict: Dictionary with mutations, CNVs, and expression (if available)
    """
    base_url = "https://www.cbioportal.org/api"
    
    headers = {"Accept": "application/json"}
    
    # Step 1: Get the sample ID from the barcode
    sample_url = f"{base_url}/samples/{barcode}"
    sample_resp = requests.get(sample_url, headers=headers)
    if sample_resp.status_code != 200:
        raise ValueError(f"Sample {barcode} not found in cBioPortal")
    
    sample_data = sample_resp.json()
    sample_id = sample_data['sampleId']
    
    # Step 2: Get genetic profiles for this study
    profiles_url = f"{base_url}/studies/{study_id}/genetic-profiles"
    profiles_resp = requests.get(profiles_url, headers=headers)
    profiles = profiles_resp.json()
    
    result = {}
    
    for profile in profiles:
        profile_id = profile['geneticProfileId']
        
        # Fetch molecular data for this sample
        molecular_url = f"{base_url}/genetic-profiles/{profile_id}/mutations/fetch"
        payload = {
            "sampleIds": [sample_id],
            "entrezGeneIds": []
        }
        mol_resp = requests.post(molecular_url, headers=headers, json=payload)
        
        if mol_resp.status_code == 200 and mol_resp.json():
            df = pd.DataFrame(mol_resp.json())
            result[profile_id] = df
    
    return result


def list_gdc_files(case_id):
    """
    List all files in GDC for a given TCGA case barcode.
    Shows file_id, file_name, data_category, data_type, workflow_type, etc.
    """
    # normalise to case-level ID
        

    filters = {
        "op": "in",
        "content": {
            "field": "cases.submitter_id",
            "value": [case_id]
        }
    }

    params = {
        "filters": str(filters).replace("'", '"'),
        "fields": "file_id,file_name,data_category,data_type,analysis.workflow_type,cases.submitter_id",
        "format": "JSON",
        "size": "2000"
    }

    r = requests.get("https://api.gdc.cancer.gov/files", params=params)
    r.raise_for_status()
    hits = r.json()["data"]["hits"]
    if not hits:
        print("No files found for this case.")
        return pd.DataFrame()

    df = pd.DataFrame(hits)
    mutation_types = [
        "Annotated Somatic Mutation",
        "Raw Simple Somatic Mutation",
        "Gene Level Copy Number",
        "Copy Number Segment",
        "Transcript Fusion"
    ]
    
    df_filtered = df[df['data_type'].isin(mutation_types)].reset_index(drop=True)
    
    def assign_analysis_type(data_type):
        if data_type in ["Annotated Somatic Mutation", "Raw Simple Somatic Mutation"]:
            return "Mutation Effect"
        elif data_type in ["Gene Level Copy Number", "Copy Number Segment"]:
            return "CNV Impact"
        elif data_type == "Transcript Fusion":
            return "Fusion Effect"
        else:
            return "Other"
    
    df_filtered['case_id'] = case_id
    df_filtered['analysis_type'] = df_filtered['data_type'].apply(assign_analysis_type)
    df_filter_SM = df_filtered[df_filtered['analysis_type'] == "Mutation Effect"]
    df_filter_CNV = df_filtered[df_filtered['analysis_type'] == "CNV Impact"]
    df_filter_FSE = df_filtered[df_filtered['analysis_type'] == "Fusion Effect"]

    return df_filter_SM, df_filter_CNV, df_filter_FSE



#take out all file_ids for further extraction of information. Since need just the case then no need do so much extra stuff...
"""
Good to note here when using this(but no need to use already)
This is for retrieving all the necessary mutational document name, for further retrieval
Do all increments of 500, but make the last 397 as (len(sample_barcodes) - indices) can already
"""
counter = 500
print(range(indices, indices + counter))
for i in range(indices, indices + counter):
    parts = sample_barcode[i].split('-')
    if len(parts) >= 3:
        case_id = "-".join(parts[:3])
    else:
        case_id = sample_barcode[i]

    if case_id in perused_caseids:
        continue

    print(case_id)
    perused_caseids.append(case_id)
    dataSM, dataCNV, dataFSE = list_gdc_files(case_id)
    # Display mutations (if any)
    if file_data_SM.empty == True and file_data_CNV.empty == True and file_data_FSE.empty == True:
        file_data_SM, file_data_CNV, file_data_FSE = dataSM.loc[:,['case_id','file_id','file_name', 'data_type', 'data_category',
        'analysis', 'analysis_type']], dataCNV.loc[:,['case_id','file_id','file_name', 'data_type', 'data_category',
        'analysis', 'analysis_type']], dataFSE.loc[:,['case_id','file_id','file_name', 'data_type', 'data_category',
        'analysis', 'analysis_type']]
    else:
        data_fil_SM, data_fil_CNV, data_fil_FSE = dataSM[file_data_SM.columns], dataCNV[file_data_CNV.columns], dataFSE[file_data_FSE.columns]
        file_data_SM,  file_data_CNV, file_data_FSE= pd.concat([file_data_SM, data_fil_SM]), pd.concat([file_data_CNV, data_fil_CNV]), pd.concat([file_data_FSE, data_fil_FSE])

indices += 500
print(file_data_SM, perused_caseids, indices)




In [2]:
#need to find a way to retrieve all information (the number of cases seem to be too much for loading - around 5.5k)
#print(SM_files.shape, file_data_CNV.shape, file_data_FSE.shape, indices, perused_caseids, sample_barcode.shape)
#saving to csv file first (already done so made it commented)
#file_data_SM.to_csv('somatic_mutation_file_GCD.csv', index=False)
#file_data_CNV.to_csv('copy_number_variation_file_GCD.csv', index=False)
#file_data_FSE.to_csv('transcript_fusion_file_GCD.csv', index=False)

#retrieving needed file databases from the saved csv files
SM_files = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/somatic_mutation_file_GCD.csv")
CNV_files = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/copy_number_variation_file_GCD.csv")
FSE_files = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/transcript_fusion_file_GCD.csv")

In [ ]:
#global variables used when checking status of files (FSE, CNV, SM)
indices2 = 0
SM_retrieve = pd.DataFrame()

In [ ]:
#checking which information i can take openly from somatic mutations.
import json
import requests

counter2 = 2000
def add_access(file_id):
    filters = {
        "op":"in",
        "content":{
            "field": "file_id",
            "value": [file_id]
        }
    }

    params = {
        "filters":json.dumps(filters),
        "fields": "file_id,access",
        "format": "JSON",
        "size": 2000
    }

    r = requests.get("https://api.gdc.cancer.gov/files", params=params)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["data"]["hits"])
    df_open = df[df['access'] == 'open']
    return df_open

print(range(indices2, indices2+counter2))
for id in range(indices2, indices2 + counter2):
    access = add_access(CNV_files['file_id'][id])
    if SM_retrieve.empty == True:
        SM_retrieve = access
    else:
        SM_retrieve = pd.concat([SM_retrieve, access])

indices2 += 2000

In [ ]:
print(CNV_files.shape, SM_retrieve.shape,indices2)
#SM_retrieve.to_csv("CNV_access_status.csv")

(6570, 7) (6570, 3) 4852


In [3]:
#merge CNV access status with copy_number_variation
CNV_access_sts = pd.read_csv("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/CNV_access_status.csv")
CNV_with_access = CNV_files.merge(CNV_access_sts, on='file_id', how='left')
#CNV_with_access.to_csv("CNV_updated_access.csv")

In [ ]:
"""
Plan now since only CNV access is open and I don't have access to GCD
1. Masked Somatic Mutation files (if have)
2. Gene Level Copy Number, Gene Expression Quantification, Clinical Supplements
3. (Optional) Allele-specific copy number segment
4. (Optional) Isoform expression quantification
(Optional) miRNA expression quantification
"""
#print(CNV_with_access['data_type'].unique())
data_types_extract = ["Masked Somatic Mutation", "Gene Level Copy Number",
    "Gene Expression Quantification", "Clinical Supplement"]

pd.options.display.max_rows = None
CNV_model_extract = CNV_with_access[CNV_with_access['data_type'].isin(data_types_extract) == True]
print(CNV_model_extract.loc[:,['case_id', 'file_id']])

In [ ]:
import os
#doing the actual retrieval of each file
"""
Note: need to see if the current function is efficient.
If data or file is still retrieved slowly then no choice la
"""
from tqdm import tqdm  # nice progress bars
from concurrent.futures import ThreadPoolExecutor

def download_gdc_file(file_id, out_path, token=None, chunk_size=8192):
    """
    Download a single file from GDC given its file_id.

    Parameters:
        file_id (str): GDC file UUID
        out_path (str): Local path where file should be saved
        token (str): (optional) path to GDC token file for controlled data
    """
    url = f"https://api.gdc.cancer.gov/data/{file_id}"
    
    headers = {}
    if token:  # if you’re downloading controlled-access data
        with open(token) as f:
            headers["X-Auth-Token"] = f.read().strip()

    resume_byte_pos = 0
    if os.path.exists(out_path):
        resume_byte_pos = os.path.getsize(out_path)
        if resume_byte_pos > 0:
            headers["Range"] = f"bytes={resume_byte_pos}-"

    # Make the request
    with requests.get(url, headers=headers, stream=True) as r:
        # If server does not support Range, status_code will be 200, else 206
        if r.status_code not in (200, 206):
            r.raise_for_status()

        # Figure out total size
        total_size = int(r.headers.get("content-length", 0))
        if "Content-Range" in r.headers:
            # If resuming, add already downloaded size
            total_size += resume_byte_pos

        mode = 'ab' if resume_byte_pos > 0 else 'wb'
        desc = f"Downloading {os.path.basename(out_path)}"
        with open(out_path, mode) as f, tqdm(
            initial=resume_byte_pos,
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc=desc
        ) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

#retrieve the file information. Must see how can i refine the command. Above seems to be just a very read write thing
def download(row):
    case_id = row['case_id']
    file_id = row['file_id']
    out_name = f"C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/CNV_gene_count_files/{case_id}_{file_id}.gz"
    #need to create a new file for each i.
    #will need to use a create csv or txt kinda thing and adapt to represent each file_id to differentiate
    #out file MUST be .maf.gz format
    download_gdc_file(file_id, out_name)

file_ids = CNV_model_extract.loc[:,['case_id','file_id']].to_dict(orient='records')
#many downloads woo
with ThreadPoolExecutor(max_workers=4) as executor:
    executor.map(download, file_ids)



In [ ]:
#doing pre-testing because retrieved files are of some weird format cb
def show_file_head(path, nbytes=200, nlines=10):
    print("Path:", path)
    try:
        with open(path, "rb") as f:
            head = f.read(nbytes)
        print("First 16 bytes (hex):", head[:16].hex())
        print("First 16 bytes (repr):", repr(head[:64]))
    except Exception as e:
        print("Could not read bytes:", e)
        return

    # try to show as text
    try:
        print("\n--- First lines (decoded, errors=replace) ---")
        with open(path, "rt", errors="replace") as f:
            for i, line in enumerate(f):
                print(line.rstrip())
                if i >= nlines - 1:
                    break
    except Exception as e:
        print("Could not decode as text (likely binary / compressed):", e)

folder = r"C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/CNV_gene_count_files"
show_file_head("C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/CNV_gene_count_files/TCGA-3C-AALI_7d49f942-01fb-4723-93cc-b640c9a8600f.gz")

In [3]:
#this one is to read gz file that is taken out from the above function
import gzip
from gprofiler import GProfiler
import zipfile


gp = GProfiler(return_dataframe=True)

def read_and_enrich(file_path, gain_threshold=4, top_n_terms=20):
    #print(file_path)
    #basename = os.path.basename(file_path)
    df = pd.read_csv(file_path, sep="\t", compression=None,low_memory=False)

    #print(f"{basename} columns: {list(df.columns)}")  # debug

    """if 'copy_number' not in df.columns or 'gene_name' not in df.columns:
        raise ValueError(f"{basename} does not have expected columns")
"""
    top_genes = df[df['copy_number'] > gain_threshold]['gene_name'] \
                    .dropna().unique().tolist()

    if not top_genes:
        # no genes above threshold
        return pd.DataFrame(columns=['source', 'name', 'p_value', 'description', 'parents'])

    enrich = gp.profile(organism='hsapiens', query=top_genes)
    #print(enrich.columns)
    # If gp.profile returns empty or missing columns, handle gracefully
    expected_cols = ['source', 'name', 'p_value', 'description', 'parents']
    available_cols = [c for c in expected_cols if c in enrich.columns]

    if not available_cols:
        # nothing useful returned, return empty frame with expected structure
        return pd.DataFrame(columns=expected_cols)

    # create a frame with expected columns; missing cols get NaN
    out = enrich.reindex(columns=expected_cols)
    return out.head(top_n_terms)

def process_gz_files(folder_path, gain_threshold=4, top_n_terms=10):
    """
    Loop through all .gz files in a folder and return a dictionary of DataFrames.

    Parameters:
        folder_path (str): Path to the folder containing .gz files.
        sep (str): Separator for CSV/TSV files. Default: tab.
        **read_csv_kwargs: Any extra arguments to pass to pandas.read_csv.

    Returns:
        dict: {filename: DataFrame} for each gz file.
    """
    results = {}
    for fname in os.listdir(folder_path):
        if not fname.endswith(".gz"):
            continue
        full_path = os.path.normpath(os.path.join(folder_path, fname))
        #print(full_path)
        print(f"Reading: {fname}")
        try:
            df = read_and_enrich(full_path, gain_threshold=4, top_n_terms=10)
            results[fname] = df
        except Exception as e:
            print(f"Error reading {fname}: {e}")
    return results

folder = r"C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/CNV_gene_count_files"
enrich_folder = Path("enrich_results")
enrich_folder.mkdir(exist_ok=True)
for fname in os.listdir(folder):
    if not fname.endswith(".gz"):
        continue
    full_path = os.path.join(folder, fname)
     
    out_file = enrich_folder / f"{fname}_enrichplus.json"
    if out_file.exists():
        print(f"Skipping {fname}, already processed.")
        continue
    print("Processing", fname)
    try:
        df_enrich = read_and_enrich(full_path)
        json_str = df_enrich.to_json(orient="records")

        # write JSON string to file
        with open(out_file, "w") as f:
            json.dump(json_str, f)
        break
    except Exception as e:
        print(f"Error reading {fname}: {e}")
#results = process_gz_files(folder_path=folder)


Processing TCGA-3C-AAAU_75cfa36d-325f-4fd3-a908-ef17980ed614.gz


In [11]:
#looking into a dataframe of one enrich results to see which ones are important to take
def load_json_to_df(url, record_path=None):
    with open(url, "r") as f:
        data = json.load(f)
    
    data = json.loads(data)

    # Now it’s a list of dicts, so safe to convert
    return pd.DataFrame(data)



pd.set_option('display.max_colwidth', None)

folder = r"C:/Users/user/Desktop/Data_Stuffs/Portfolio_Projects/ML_AI_Models/Oxidative_Stress_Cancer/enrich_results"
enrich_folder = Path("enrich_results")
enrich_folder.mkdir(exist_ok=True)
count = 0
for fname in os.listdir(folder):
    case_id = fname[:12]
    enrich = load_json_to_df("enrich_results/" + fname)
    enrich['case_id'] = case_id
    enrich.to_csv("enrich_data.csv", mode='a', index=False, header=False)

"""enrich1 = load_json_to_df("enrich_results/TCGA-3C-AAAU_75cfa36d-325f-4fd3-a908-ef17980ed614.gz_enrich.json")
enrich2 = load_json_to_df("enrich_results/TCGA-3C-AAAU_75cfa36d-325f-4fd3-a908-ef17980ed614.gz_enrichplus.json")
print(enrich1[["parents", 'source', 'description']])
enrich2.loc[:,["parents", 'source', 'description']]"""

'enrich1 = load_json_to_df("enrich_results/TCGA-3C-AAAU_75cfa36d-325f-4fd3-a908-ef17980ed614.gz_enrich.json")\nenrich2 = load_json_to_df("enrich_results/TCGA-3C-AAAU_75cfa36d-325f-4fd3-a908-ef17980ed614.gz_enrichplus.json")\nprint(enrich1[["parents", \'source\', \'description\']])\nenrich2.loc[:,["parents", \'source\', \'description\']]'

In [ ]:
#doing routine saving of dataframes. Too long man this one.
gprofiler_json = {
    df.to_json(orient="records")
    for name, df in results.items()
}

with open("gprofiler_data.json", "w") as f:
    json.dump(gprofiler_json, f)